
# 📊 Telecom X — Análise Exploratória de Dados (EDA)
## Projeto: Churn de Clientes

Este notebook tem como objetivo realizar uma **Análise Exploratória de Dados (EDA)** sobre os dados de clientes da Telecom X,
buscando identificar padrões, tendências e fatores associados ao **cancelamento de clientes (churn)**.

---
### Objetivos do EDA
- Entender o perfil dos clientes
- Avaliar a distribuição do churn
- Identificar variáveis mais associadas ao cancelamento
- Gerar insights para apoiar modelos preditivos e estratégias de retenção



## 1. Importação das Bibliotecas


In [ ]:

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)



## 2. Extração dos Dados
Os dados foram obtidos a partir de uma API e disponibilizados em formato JSON.


In [ ]:

path = "TelecomX_Data.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data)
df.head()



## 3. Transformação e Limpeza dos Dados
Nesta etapa:
- Normalizamos os nomes das colunas
- Convertimos variáveis numéricas
- Tratamos valores ausentes
- Criamos a variável alvo binária (churn)


In [ ]:

# Ajustando nomes das colunas
df.columns = df.columns.str.replace('.', '_')

# Convertendo Total Charges para numérico
df['account_Charges_Total'] = pd.to_numeric(df['account_Charges_Total'], errors='coerce')

# Tratando churn vazio
df['Churn'] = df['Churn'].replace('', np.nan)

# Criando variável binária
df['Churn_bin'] = df['Churn'].map({'Yes': 1, 'No': 0})

df.info()



## 4. Estrutura do Dataset e Dicionário de Dados

Após a extração e normalização dos dados, é fundamental compreender a estrutura do dataset e o significado de cada variável,
conectando o dado técnico ao contexto de negócio e à análise de evasão de clientes (churn).

### Tipos de Variáveis

- **Identificador**
  - `customerID`: identificador único do cliente (não utilizado em modelos)

- **Variável alvo**
  - `Churn`: indica se o cliente cancelou o serviço
  - `Churn_bin`: versão binária (1 = cancelou, 0 = permaneceu)

- **Variáveis demográficas**
  - `customer_gender`
  - `customer_SeniorCitizen`
  - `customer_Partner`
  - `customer_Dependents`

- **Relacionamento com a empresa**
  - `customer_tenure`: tempo de contrato em meses
  - `account_Contract`: tipo de contrato

- **Serviços contratados**
  - `phone_PhoneService`, `phone_MultipleLines`
  - `internet_InternetService`
  - `internet_OnlineSecurity`, `internet_OnlineBackup`
  - `internet_DeviceProtection`, `internet_TechSupport`
  - `internet_StreamingTV`, `internet_StreamingMovies`

- **Faturamento e pagamento**
  - `account_PaperlessBilling`
  - `account_PaymentMethod`
  - `account_Charges_Monthly`
  - `account_Charges_Total`

### Relevância das Variáveis para Churn

- **Alta relevância**
  - `customer_tenure`
  - `account_Contract`
  - `account_Charges_Monthly`

- **Média relevância**
  - Serviços adicionais de suporte e segurança
  - Forma de pagamento

- **Baixa relevância**
  - Variáveis puramente demográficas

### Padronização e Transformação

Nesta análise, algumas transformações foram aplicadas para melhorar a clareza e facilitar análises futuras:

- Padronização dos nomes das colunas
- Conversão de valores textuais ("Yes"/"No") em formato binário
- Conversão de variáveis financeiras para formato numérico

Essas etapas tornam o dataset mais consistente, interpretável e preparado para modelagem preditiva.



## 4. Visão Geral dos Dados


In [ ]:

df.describe(include='all')



## 5. Análise da Variável Alvo (Churn)


In [ ]:

df['Churn'].value_counts(dropna=False)


In [ ]:

plt.figure()
df['Churn'].value_counts(dropna=False).plot(kind='bar')
plt.title("Distribuição de Churn")
plt.xlabel("Churn")
plt.ylabel("Quantidade")
plt.show()



**Insight:**  
A maioria dos clientes não cancela, porém há um volume relevante de churn.
Também existem registros sem informação de churn, que devem ser tratados antes da modelagem.



## 6. Churn por Tipo de Contrato


In [ ]:

churn_contract = (
    df.groupby('account_Contract')['Churn_bin']
    .mean()
    .sort_values(ascending=False)
)

churn_contract


In [ ]:

plt.figure()
churn_contract.plot(kind='bar')
plt.title("Taxa Média de Churn por Tipo de Contrato")
plt.xlabel("Tipo de Contrato")
plt.ylabel("Taxa de Churn")
plt.show()



**Insight:**  
Clientes com contrato *Month-to-month* apresentam taxa de churn significativamente maior.
Contratos de longo prazo reduzem drasticamente o risco de cancelamento.



## 7. Tempo de Permanência (Tenure) x Churn


In [ ]:

df.groupby('Churn')['customer_tenure'].mean()


In [ ]:

plt.figure()
df.groupby('Churn')['customer_tenure'].mean().plot(kind='bar')
plt.title("Tempo Médio de Contrato por Churn")
plt.xlabel("Churn")
plt.ylabel("Tenure Médio (meses)")
plt.show()



**Insight:**  
Clientes que cancelam possuem, em média, muito menos tempo de contrato.
Os primeiros meses são críticos para retenção.



## 8. Serviços e Churn


In [ ]:

services = [
    'internet_OnlineSecurity',
    'internet_OnlineBackup',
    'internet_DeviceProtection',
    'internet_TechSupport',
    'internet_StreamingTV',
    'internet_StreamingMovies'
]

for col in services:
    print(f"\n{col}")
    print(df.groupby(col)['Churn_bin'].mean())



**Insight:**  
Clientes que não possuem serviços adicionais de suporte e segurança tendem a apresentar maior churn.
Esses serviços podem ser utilizados como alavancas de retenção.



## 9. Conclusões do EDA
- Contrato é o fator mais relevante para churn
- Clientes novos apresentam maior risco
- Serviços agregados reduzem a evasão
- Dataset está pronto para modelos preditivos após ajustes finais

👉 Próximo passo: **Feature Engineering e Modelagem Preditiva**



## 10. Análise de Correlação entre Variáveis

Como etapa adicional da Análise Exploratória de Dados, avaliamos a **correlação entre variáveis**
para entender quais fatores apresentam maior relação com a evasão de clientes (churn).

Essa análise apoia:
- Seleção de variáveis para modelos preditivos
- Criação de novas features
- Interpretação do comportamento do cliente

### Hipótese analisada
Clientes que contratam mais serviços tendem a apresentar menor probabilidade de churn.


In [ ]:

service_cols = [
    'phone_PhoneService',
    'phone_MultipleLines',
    'internet_OnlineSecurity',
    'internet_OnlineBackup',
    'internet_DeviceProtection',
    'internet_TechSupport',
    'internet_StreamingTV',
    'internet_StreamingMovies'
]

for col in service_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df['Total_Services'] = df[service_cols].sum(axis=1)
df[['Total_Services', 'Churn_bin']].head()


In [ ]:

plt.figure()
plt.scatter(df['Total_Services'], df['Churn_bin'])
plt.title("Quantidade de Serviços Contratados vs Churn")
plt.xlabel("Total de Serviços")
plt.ylabel("Churn (0 = Não, 1 = Sim)")
plt.show()


In [ ]:

corr_vars = [
    'Churn_bin',
    'customer_tenure',
    'account_Charges_Monthly',
    'account_Charges_Total',
    'Total_Services'
]

corr_matrix = df[corr_vars].corr()
corr_matrix


In [ ]:

plt.figure()
plt.imshow(corr_matrix)
plt.colorbar()
plt.xticks(range(len(corr_vars)), corr_vars, rotation=45)
plt.yticks(range(len(corr_vars)), corr_vars)
plt.title("Matriz de Correlação entre Variáveis")
plt.show()



### Insights da Análise de Correlação

- Clientes com **maior quantidade de serviços** apresentam menor churn.
- **Tenure** possui forte correlação negativa com churn.
- **Charges.Monthly** apresenta correlação positiva com churn.
- **Charges.Total** reflete clientes mais antigos e estáveis.

Essa análise reforça a importância de variáveis de engajamento e relacionamento.
